In [1]:
!pip install transformers accelerate bitsandbytes gradio torch sentencepiece datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00


In [2]:
# ============================================================
# SQL Query Generator using SQLCoder-7B-2 + Gradio Interface
# ============================================================

# Imports ──────────────────────────────────────────
import torch
import gradio as gr
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from datasets import load_dataset
import json
import re


In [3]:
#  Configuration ────────────────────────────────────
MODEL_NAME = "defog/sqlcoder-7b-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_4BIT = torch.cuda.is_available()  # Use 4-bit quantization if GPU available

print(f"Using device: {DEVICE}")
print(f"4-bit quantization: {USE_4BIT}")


Using device: cuda
4-bit quantization: True


In [4]:
# Load Spider Dataset (schema extraction) ──────────
def load_spider_schemas():
    """Load Spider dataset and extract database schemas."""
    print("Loading Spider dataset...")
    dataset = load_dataset("xlangai/spider", trust_remote_code=True)

    # Build a dict: db_name -> schema DDL
    schemas = {}

    # Spider tables.json has the schema info
    # We'll build sample schemas from the dataset
    train_data = dataset["train"]

    db_schemas = {}
    for item in train_data:
        db_id = item["db_id"]
        if db_id not in db_schemas:
            db_schemas[db_id] = {
                "question_examples": [],
                "query_examples": []
            }
        if len(db_schemas[db_id]["question_examples"]) < 3:
            db_schemas[db_id]["question_examples"].append(item["question"])
            db_schemas[db_id]["query_examples"].append(item["query"])

    return db_schemas, dataset

In [5]:
#  Predefined Schemas (for demo without full Spider) ─
SAMPLE_SCHEMAS = {
    "ecommerce": """
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100),
    city VARCHAR(50),
    country VARCHAR(50),
    signup_date DATE
);

CREATE TABLE products (
    product_id INT PRIMARY KEY,
    name VARCHAR(100),
    category VARCHAR(50),
    price DECIMAL(10, 2),
    stock_quantity INT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT REFERENCES customers(customer_id),
    order_date DATE,
    total_amount DECIMAL(10, 2),
    status VARCHAR(20)
);

CREATE TABLE order_items (
    item_id INT PRIMARY KEY,
    order_id INT REFERENCES orders(order_id),
    product_id INT REFERENCES products(product_id),
    quantity INT,
    unit_price DECIMAL(10, 2)
);
""",
    "university": """
CREATE TABLE students (
    student_id INT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    major VARCHAR(50),
    gpa DECIMAL(3, 2),
    enrollment_year INT
);

CREATE TABLE courses (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100),
    department VARCHAR(50),
    credits INT,
    instructor VARCHAR(100)
);

CREATE TABLE enrollments (
    enrollment_id INT PRIMARY KEY,
    student_id INT REFERENCES students(student_id),
    course_id INT REFERENCES courses(course_id),
    grade VARCHAR(2),
    semester VARCHAR(20)
);
""",
    "hospital": """
CREATE TABLE patients (
    patient_id INT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    gender VARCHAR(10),
    blood_type VARCHAR(5),
    admission_date DATE
);

CREATE TABLE doctors (
    doctor_id INT PRIMARY KEY,
    name VARCHAR(100),
    specialty VARCHAR(50),
    department VARCHAR(50),
    years_experience INT
);

CREATE TABLE appointments (
    appointment_id INT PRIMARY KEY,
    patient_id INT REFERENCES patients(patient_id),
    doctor_id INT REFERENCES doctors(doctor_id),
    appointment_date DATE,
    diagnosis VARCHAR(200),
    treatment VARCHAR(200)
);

CREATE TABLE medications (
    medication_id INT PRIMARY KEY,
    appointment_id INT REFERENCES appointments(appointment_id),
    drug_name VARCHAR(100),
    dosage VARCHAR(50),
    duration_days INT
);
""",
    "custom": ""  # User will provide their own
}

# Sample questions for each schema
SAMPLE_QUESTIONS = {
    "ecommerce": [
        "Show me the top 5 customers by total order amount",
        "Which products are low in stock (less than 10 units)?",
        "What is the total revenue per product category?",
        "List all orders placed in the last 30 days with customer names",
        "Find customers who have never placed an order",
    ],
    "university": [
        "List all students with GPA above 3.5",
        "Which course has the most enrollments?",
        "Show average GPA per major",
        "Find students who are enrolled in more than 3 courses",
        "List courses taught by each instructor",
    ],
    "hospital": [
        "How many patients were admitted per month this year?",
        "Which doctor has treated the most patients?",
        "List all patients currently on more than 2 medications",
        "Find the most common diagnoses",
        "Show all appointments for patients above 60 years old",
    ],
    "custom": []
}


In [6]:
#Load SQLCoder Model
def load_model():
    """Load SQLCoder-7B-2 with optional 4-bit quantization."""
    print(f"Loading {MODEL_NAME}...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    if USE_4BIT:
        # 4-bit quantization for memory efficiency on GPU
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        # CPU fallback (slower)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            device_map="cpu",
            trust_remote_code=True,
        )

    print("Model loaded successfully!")
    return tokenizer, model

In [7]:
# Prompt Template (SQLCoder format) ─────────────────
def build_prompt(question: str, schema: str, dialect: str = "SQL") -> str:
    """Build the SQLCoder prompt using its expected format."""
    prompt = f"""### Task
Generate a {dialect} query to answer [QUESTION]{question}[/QUESTION]

### Instructions
- If you cannot answer the question with the available database schema, return 'I do not know'
- Remember that revenue is price multiplied by quantity
- Remember that 1 means true and 0 means false in the database
- Ensure all columns referenced are present in the schema

### Database Schema
The query will run on a database with the following schema:
{schema}

### Answer
Given the database schema, here is the SQL query that answers [QUESTION]{question}[/QUESTION]
[SQL]"""
    return prompt


In [8]:
# SQL Generation Function ──────────────────────────
def generate_sql(
    tokenizer,
    model,
    question: str,
    schema: str,
    dialect: str = "SQL",
    max_new_tokens: int = 300,
    temperature: float = 0.1,
    num_beams: int = 4,
) -> str:
    """Generate SQL query from natural language question."""

    if not question.strip():
        return " Please enter a question."
    if not schema.strip():
        return " Please provide a database schema."

    prompt = build_prompt(question, schema, dialect)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            num_beams=num_beams,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens (not the prompt)
    generated = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    # Extract just the SQL part
    sql = generated.strip()

    # Clean up end tokens
    for stop in ["[/SQL]", "###", "\n\n\n"]:
        if stop in sql:
            sql = sql[:sql.index(stop)]

    return sql.strip()

In [9]:
#  Gradio Interface ──────────────────────────────────
def build_gradio_app(tokenizer, model):
    """Build and launch the Gradio interface."""

    def on_schema_select(schema_name):
        """Update schema and sample questions when DB is selected."""
        schema = SAMPLE_SCHEMAS.get(schema_name, "")
        questions = SAMPLE_QUESTIONS.get(schema_name, [])
        sample_text = "\n".join([f"• {q}" for q in questions]) if questions else ""
        return schema, sample_text

    def on_generate(question, schema_text, dialect, max_tokens, temperature):
        """Handle SQL generation."""
        if not question.strip():
            return "Please enter a question.", ""
        if not schema_text.strip():
            return "Please provide or select a schema.", ""

        try:
            sql = generate_sql(
                tokenizer, model,
                question=question,
                schema=schema_text,
                dialect=dialect,
                max_new_tokens=int(max_tokens),
                temperature=float(temperature),
            )

            # Format nicely
            formatted = f"```sql\n{sql}\n```"
            explanation = f" Generated {dialect} query for: \"{question}\""
            return formatted, explanation

        except Exception as e:
            return f" Error: {str(e)}", ""

    # ── Gradio UI Layout ──────────────────────────────────────
    with gr.Blocks(
        title="SQL Query Generator — SQLCoder-7B-2",
        theme=gr.themes.Soft(),
        css="""
        .header { text-align: center; margin-bottom: 20px; }
        .sql-output { font-family: monospace; }
        """
    ) as demo:

        # Header
        gr.Markdown("""
        #  SQL Query Generator
        ### Powered by [SQLCoder-7B-2](https://huggingface.co/defog/sqlcoder-7b-2) (defog/sqlcoder-7b-2)
        Convert natural language questions into accurate SQL queries.
        ---
        """)

        with gr.Row():
            # ── Left Column: Input ────────────────────────────
            with gr.Column(scale=1):
                gr.Markdown("###  Configuration")

                schema_selector = gr.Dropdown(
                    choices=list(SAMPLE_SCHEMAS.keys()),
                    value="ecommerce",
                    label="📂 Select Sample Database",
                    info="Choose a pre-built schema or select 'custom'"
                )

                sample_questions_box = gr.Textbox(
                    label="💡 Sample Questions for This Schema",
                    lines=5,
                    interactive=False,
                    value="\n".join([f"• {q}" for q in SAMPLE_QUESTIONS["ecommerce"]])
                )

                dialect = gr.Radio(
                    choices=["SQL", "MySQL", "PostgreSQL", "SQLite", "BigQuery"],
                    value="SQL",
                    label=" SQL Dialect"
                )

                with gr.Accordion(" Advanced Settings", open=False):
                    max_tokens = gr.Slider(
                        minimum=100, maximum=500, value=300, step=50,
                        label="Max New Tokens"
                    )
                    temperature = gr.Slider(
                        minimum=0.0, maximum=1.0, value=0.1, step=0.05,
                        label="Temperature (lower = more deterministic)"
                    )

            # ── Right Column: Schema + Query ──────────────────
            with gr.Column(scale=2):
                gr.Markdown("###  Database Schema (DDL)")
                schema_input = gr.Textbox(
                    value=SAMPLE_SCHEMAS["ecommerce"],
                    lines=15,
                    label="Schema (CREATE TABLE statements)",
                    placeholder="Paste your CREATE TABLE statements here...",
                    elem_classes=["sql-output"]
                )

                gr.Markdown("###  Your Question")
                question_input = gr.Textbox(
                    lines=2,
                    label="Natural Language Question",
                    placeholder="e.g. Show me the top 5 customers by total order amount",
                )

                generate_btn = gr.Button(
                    " Generate SQL Query",
                    variant="primary",
                    size="lg"
                )

        # ── Output Section ────────────────────────────────────
        gr.Markdown("---\n###  Generated SQL Query")

        status_text = gr.Textbox(
            label="Status",
            interactive=False,
            lines=1
        )

        sql_output = gr.Textbox(
            label="SQL Query",
            lines=10,
            interactive=False,
            elem_classes=["sql-output"]
        )

        copy_btn = gr.Button(" Copy SQL", size="sm")

        # ── Examples ──────────────────────────────────────────
        gr.Markdown("---\n###  Quick Examples")
        gr.Examples(
            examples=[
                ["Show me the top 5 customers by total order amount", SAMPLE_SCHEMAS["ecommerce"], "SQL"],
                ["Which products are low in stock (less than 10 units)?", SAMPLE_SCHEMAS["ecommerce"], "SQL"],
                ["List all students with GPA above 3.5", SAMPLE_SCHEMAS["university"], "PostgreSQL"],
                ["How many patients were admitted per month this year?", SAMPLE_SCHEMAS["hospital"], "MySQL"],
            ],
            inputs=[question_input, schema_input, dialect],
        )

        # ── Event Handlers ────────────────────────────────────
        schema_selector.change(
            fn=on_schema_select,
            inputs=[schema_selector],
            outputs=[schema_input, sample_questions_box]
        )

        generate_btn.click(
            fn=on_generate,
            inputs=[question_input, schema_input, dialect, max_tokens, temperature],
            outputs=[sql_output, status_text]
        )

        question_input.submit(
            fn=on_generate,
            inputs=[question_input, schema_input, dialect, max_tokens, temperature],
            outputs=[sql_output, status_text]
        )

    return demo



In [10]:

#  Main Entry Point ──────────────────────────────────
if __name__ == "__main__":
    # Load the model
    tokenizer, model = load_model()

    # Build and launch Gradio app
    demo = build_gradio_app(tokenizer, model)

    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,          # Set True to get a public URL
        show_error=True,
    )

Loading defog/sqlcoder-7b-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully!


/tmp/ipykernel_3808/435634776.py:38: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3808/435634776.py:38: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>